# Trabajo Nº2 – Clasificación de Spam y No Spam con NLP

Este notebook sirve como plantilla mínima para que puedas comenzar a construir tus propios modelos de clasificación de texto.  
Incluye:
- Configuración básica e imports.
- Carga de los datasets `train.csv` y `test.csv` desde la carpeta `Spam_Data/`.
- Celda final para generar el archivo `.csv` de predicciones con el formato requerido.

Indicaciones:
- En la entrega se proporciona `Spam_Data.zip`, que contiene:
  - `train.csv` (con la columna `spam_label` como variable objetivo)
  - `test.csv` (sin etiquetas)
  - `sample_submission.csv` (ejemplo del formato del .csv)
- Descomprime el zip en el mismo directorio del notebook antes de ejecutar.
- Tú debes definir, entrenar y comparar tus modelos.
- Al final, genera `submission.csv` con el formato especificado para la entrega.

In [20]:
# Dependencias para manipular datos, graficar y entrenar modelos de NLP
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn import metrics
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import text as sk_text

# Configuración de visualización y semillas para reproducibilidad
pd.set_option('display.max_rows', 36)
pd.set_option("display.max_colwidth", 150)
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(33)
tf.random.set_seed(33)


## Carga de Datos

Se cargan los conjuntos **train** y **test** desde la carpeta `dataset`, asumiendo esta estructura:

```text
dataset/
├─ train.csv
├─ test.csv
├─ sample_submission.csv
```

Contenido:

- `train.csv` contiene las columnas **`row_id`**, **`text`** y **`spam_label`** (0 = NO SPAM, 1 = SPAM).
- `test.csv` contiene las columnas **`row_id`** y **`text`** (sin etiquetas).
- `sample_submission.csv` muestra el formato de entrega requerido.

Las etiquetas se almacenan como enteros:

- `0` → NO SPAM
- `1` → SPAM

In [7]:
# Ruta base del conjunto de datos
DATASET_DIR = "dataset"  # Cambia si usas otra ruta

# Cargar los archivos de entrenamiento con el índice correcto
train = pd.read_csv(f"{DATASET_DIR}/train.csv", index_col="row_id")
train.head()


,text,spam_label
row_id,,
0,"You are everywhere dirt, on the floor, the windows, even on my shirt. And sometimes when i open my mouth, you are all that comes flowing out. I dr...",0
1,"Subject: \nmon , 24 may 2004 12 : 14 : 06 - 0600\ni am taking the liberty of writing you this\nletter instead of interrupting you\nby phone . plea...",1
2,So the sun is anti sleep medicine.,0
3,Hey are you angry with me. Reply me dr.,0
4,Ü go home liao? Ask dad to pick me up at 6...,0


In [8]:
# Cargar los textos de prueba que se usarán para generar la entrega
test = pd.read_csv(f"{DATASET_DIR}/test.csv", index_col="row_id")
test.head()


,text
row_id,
7097,"Subject: king ranch processed volumes at tailgate\nd .\nmary provided this to brian , please check against your numbers asap\n- - - - - - - - - - ..."
7098,"Subject: cialis , viagra , xanax , valium at low price ! no prescription needed !\ndiscount rx is simple , quick , and affordable ! br\noffering m..."
7099,"Subject: september deal inactivation in sitara\ncurrently , prior month deals are inactivated in sitara for portfolio\npurposes on the 10 th of th..."
7100,Subject: jan . 01 sale to texas general land office\nlinda -\ni did not know that this was part of a natural gas / crude oil exchange deal\nand bi...
7101,Subject: 5 th changes @ duke and air liquide\n- - - - - - - - - - - - - - - - - - - - - - forwarded by ami chokshi / corp / enron on 02 / 04 / 200...


## Desarrollo de la Práctica

### Exploración breve del conjunto

Antes de vectorizar, se revisa la distribución de clases y algunos ejemplos para comprobar el balance y la calidad de los textos.

In [9]:
# Revisar la proporción de clases para conocer el balance del conjunto
train[['spam_label']].value_counts(normalize=True)

# Mostrar algunos ejemplos aleatorios para inspeccionar el texto sin procesar
train.sample(5, random_state=33)


,text,spam_label
row_id,,
1644,Not a lot has happened here. Feels very quiet. Beth is at her aunts and charlie is working lots. Just me and helen in at the mo. How have you been?,0
5716,I've reached home finally...,0
895,"I frequently have business meetings in Downtown Chicago and find that the Hotel Monaco Chicago, gives me the peace of mind to make those meetings ...",1
3192,WIN URGENT! Your mobile number has been awarded with a £2000 prize GUARANTEED call 09061790121 from land line. claim 3030 valid 12hrs only 150ppm,1
3714,"My first stay at this hotel and was expecting good things based on reviews. For some reason, I kept being disappointed at every turn. Noisy room o...",0


### Preprocesamiento ligero y partición

Se aplica una limpieza mínima (minúsculas, eliminación de caracteres no alfanuméricos y stopwords en inglés) y se genera una partición estratificada para validar el modelo.

In [10]:
# Conjunto de stopwords en inglés para limpiar palabras poco informativas
stopwords = sk_text.ENGLISH_STOP_WORDS

def limpiar_texto(texto: str) -> str:
    # Convertir a minúsculas y eliminar signos de puntuación
    texto = texto.lower()
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    # Filtrar stopwords y recomponer la cadena limpia
    palabras = [t for t in texto.split() if t not in stopwords]
    return ' '.join(palabras)

# Crear una columna con el texto ya normalizado
train['text_clean'] = train['text'].apply(limpiar_texto)

# Separar datos en entrenamiento y validación manteniendo la proporción de clases
X_train, X_val, y_train, y_val = train_test_split(
    train['text_clean'], train['spam_label'], test_size=0.2, random_state=33, stratify=train['spam_label']
)
print(f'Tamaño train: {X_train.shape[0]}, validación: {X_val.shape[0]}')


Tamaño train: 5677, validación: 1420


### Vectorización TF–IDF y modelo lineal

Se construye un pipeline con `TfidfVectorizer` y `LogisticRegression`, manteniendo hiperparámetros moderados para evaluar un modelo lineal rápido.

In [11]:
# Pipeline con vectorización TF-IDF y un clasificador lineal balanceado
pipeline = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=200, n_jobs=-1, class_weight='balanced', random_state=33)),
])

# Entrenar sobre el conjunto de entrenamiento reducido
pipeline.fit(X_train, y_train)

# Evaluar en validación con varias métricas para tener una visión global
y_val_pred = pipeline.predict(X_val)

resultados = {
    'accuracy': metrics.accuracy_score(y_val, y_val_pred),
    'precision': metrics.precision_score(y_val, y_val_pred),
    'recall': metrics.recall_score(y_val, y_val_pred),
    'f1': metrics.f1_score(y_val, y_val_pred),
    'mcc': matthews_corrcoef(y_val, y_val_pred),
}
resultados


{'accuracy': 0.9154929577464789,
 'precision': 0.8051948051948052,
 'recall': 0.8732394366197183,
 'f1': 0.8378378378378378,
 'mcc': np.float64(0.7819967492686539)}

### Entrenamiento final y predicción sobre el test

Con el pipeline validado se ajusta sobre todo el conjunto de entrenamiento y se generan predicciones sobre el test para reutilizar la celda de exportación.

In [13]:
# Reentrenar el pipeline con todos los datos etiquetados
pipeline_final = pipeline
pipeline_final.fit(train['text_clean'], train['spam_label'])

# Aplicar el mismo preprocesado al conjunto de prueba
test['text_clean'] = test['text'].apply(limpiar_texto)

# Generar predicciones finales y revisar la distribución de clases prevista
y_pred = pipeline_final.predict(test['text_clean'])

unique, counts = np.unique(y_pred, return_counts=True)
print(dict(zip(unique, counts)))


{np.int64(0): np.int64(3382), np.int64(1): np.int64(1350)}


## Generación del CSV final de Predicciones

Una vez **definido y entrenado tu modelo** (no incluido en esta plantilla) y generadas tus predicciones sobre el conjunto de prueba `test`, crea el archivo `submission.csv` con el siguiente formato:

```text
row_id,spam_label
7097,0
7098,0
7099,1
...
````

Donde:

* `row_id` es el identificador del texto en el conjunto de test.
* `spam_label` es `0` para **NO SPAM** y `1` para **SPAM**.

In [14]:
# Cargar el archivo de ejemplo de envío
submission = pd.read_csv(f"{DATASET_DIR}/sample_submission.csv")

# Reemplazar la columna con las predicciones finales del modelo
submission["spam_label"] = y_pred

# Guardar el archivo en el formato requerido para la plataforma
output_path = f"{DATASET_DIR}/submission.csv"
submission.to_csv(output_path, index=False)

print(f"Archivo guardado correctamente como: {output_path}")
submission.head()


Archivo guardado correctamente como: dataset/submission.csv


,row_id,spam_label
0,7097,0
1,7098,1
2,7099,0
3,7100,0
4,7101,0
